In [1]:
foundry_name = ! terraform output -raw foundry_name
foundry_name = foundry_name.n
print("Foundry Name: ", foundry_name)

foundry_project_name = ! terraform output -raw foundry_project_name
foundry_project_name = foundry_project_name.n
print("Foundry Project Name: ", foundry_project_name)

foundry_project_endpoint = ! terraform output -raw foundry_project_endpoint
foundry_project_endpoint = foundry_project_endpoint.n
print("Foundry Project Endpoint: ", foundry_project_endpoint)

llm_model_deployment_name = ! terraform output -raw llm_model_deployment_name
llm_model_deployment_name = llm_model_deployment_name.n
print("LLM Model Deployment Name: ", llm_model_deployment_name)

acr_name = ! terraform output -raw acr_name
acr_name = acr_name.n
print("ACR Name: ", acr_name)

hosted_agent_name = "foundry-hosted-agent"
hosted_agent_version = "1.0.3"
agent_container = f"{acr_name}.azurecr.io/{hosted_agent_name}:{hosted_agent_version}"

Foundry Name:  foundry-650
Foundry Project Name:  foundry-project-650
Foundry Project Endpoint:  https://foundry-650.services.ai.azure.com/api/projects/foundry-project-650
LLM Model Deployment Name:  gpt-5.2
ACR Name:  acr4hostedagent650


Login to Azure Container Registry (ACR) and build the image container in ACR. You don't need to install Docker runtime in your machine. Use the `Dockerfile` within the source code to build the image.

In [2]:
# Login to ACR
! az acr login -n {acr_name} --expose-token

# Build image container image for hosted agent inside ACR
! az acr build -r {acr_name} -t {hosted_agent_name}:{hosted_agent_version} --platform linux/amd64 ./agent_simple_hosted/

{
  "accessToken": "eyJhbGciOiJSUzI1NiIsInR5cCI6IkpXVCIsImtpZCI6IkxHSEY6Q0NLNDpDVVRaOlQzWUc6UFc2Mjo1UEFaOjJST086TjNZVTpOUUVaOkdXWDU6VDRZUjpVR0JXIn0.eyJqdGkiOiIwNjIzNDFlYi1lNmFmLTQyZDctOTliMi02ZjA2MTQyYjc2ZWYiLCJzdWIiOiJhZG1pbkBNbmdFbnZNQ0FQNzg0NjgzLm9ubWljcm9zb2Z0LmNvbSIsIm5iZiI6MTc3NDM1Nzc3OCwiZXhwIjoxNzc0MzY5NDc4LCJpYXQiOjE3NzQzNTc3NzgsImlzcyI6IkF6dXJlIENvbnRhaW5lciBSZWdpc3RyeSIsImF1ZCI6ImFjcjRob3N0ZWRhZ2VudDY1MC5henVyZWNyLmlvIiwidmVyc2lvbiI6IjEuMCIsInJpZCI6IjkxYmY5NjE4NDA0ZDRkODE4YzMyNGQ2NDhkNWQ1MWViIiwiZ3JhbnRfdHlwZSI6InJlZnJlc2hfdG9rZW4iLCJhcHBpZCI6IjA0YjA3Nzk1LThkZGItNDYxYS1iYmVlLTAyZjllMWJmN2I0NiIsInRlbmFudCI6IjkzMTM5ZDFlLWEzYzEtNGQ3OC05ZWQ1LTg3OGJlMDkwZWJhNCIsInBlcm1pc3Npb25zIjp7ImFjdGlvbnMiOlsicmVhZCIsIndyaXRlIiwiZGVsZXRlIiwibWV0YWRhdGEvcmVhZCIsIm1ldGFkYXRhL3dyaXRlIiwiZGVsZXRlZC9yZWFkIiwiZGVsZXRlZC9yZXN0b3JlL2FjdGlvbiJdfSwicm9sZXMiOltdfQ.FVo-QwxlTWLZJ3uGOpIchEhmOs2siCoxAAWidJRVWxO2PgtDUsiBH-941694YHeRmzXoPKqHZJeoiQkwR2or35hbQaCXjhjBAt9ihTz0q3tPBGGSc2i5gZR35ekKDGRDb8qhzjIibmcDE

ERROR: './agent_simple_hosted/' doesn't exist.


In [2]:
# %pip install azure-ai-projects
%pip install azure-ai-agentserver-agentframework
# azure-ai-agentserver-core azure-ai-agentserver-agentframework

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [25]:
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import ImageBasedHostedAgentDefinition, ProtocolVersionRecord, AgentProtocol
from azure.identity import AzureCliCredential #  DefaultAzureCredential, 

# Initialize the client
client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=AzureCliCredential() # DefaultAzureCredential()
)

# Create the agent from a container image
agent = client.agents.create_version(
    agent_name=hosted_agent_name,
    definition=ImageBasedHostedAgentDefinition(
        container_protocol_versions=[ProtocolVersionRecord(protocol=AgentProtocol.RESPONSES, version=hosted_agent_version)],
        cpu="1",
        memory="2Gi",
        image=agent_container,
        environment_variables={
            "AZURE_AI_PROJECT_ENDPOINT": foundry_project_endpoint,
            "AZURE_AI_MODEL_DEPLOYMENT_NAME": llm_model_deployment_name,
        }
    )
)

In [27]:
print("Hosted Agent name:", agent.name)

print("Hosted Agent version:", agent.version)

Hosted Agent name: foundry-hosted-agent
Hosted Agent version: 4


## Start an agent deployment

After you create your hosted agent version, you can start the deployment by using the az CLI extension to make it available for requests. You can also start a hosted agent that's stopped.

In [28]:
! az cognitiveservices agent start --account-name {foundry_name} --project-name {foundry_project_name} --name {hosted_agent_name} --agent-version {agent.version}

! az cognitiveservices agent show --account-name {foundry_name} --project-name {foundry_project_name} --name {hosted_agent_name}

{
  "agent_id": "foundry-hosted-agent",
  "agent_version_id": "4",
  "container": {
    "created_at": "2026-02-26T14:27:51.5997004Z",
    "id": "foundry-hosted-agent-4",
    "max_replicas": 1,
    "min_replicas": 1,
    "object": "agent.container",
    "status": "Starting",
    "updated_at": "2026-02-26T14:27:51.5997008Z"
  },
  "id": "8886c689-48be-41b3-8001-a0786d02458f",
  "status": "InProgress"
}


{
  "id": "foundry-hosted-agent",
  "name": "foundry-hosted-agent",
  "object": "agent",
  "versions": {
    "latest": {
      "created_at": 1772115999,
      "definition": {
        "container_protocol_versions": [
          {
            "protocol": "responses",
            "version": "1.0.3"
          }
        ],
        "cpu": "1",
        "environment_variables": {
          "AZURE_AI_MODEL_DEPLOYMENT_NAME": "gpt-5.2",
          "AZURE_AI_PROJECT_ENDPOINT": "https://foundry-650.services.ai.azure.com/api/projects/foundry-project-650"
        },
        "image": "acr4hostedagent650.azurecr.io/foundry-hosted-agent:1.0.3",
        "kind": "hosted",
        "memory": "2Gi"
      },
      "description": "",
      "id": "foundry-hosted-agent:4",
      "metadata": {},
      "name": "foundry-hosted-agent",
      "object": "agent.version",
      "version": "4"
    }
  }
}


If starting the hosted agent using the above command line fails, you can start it using the Foundry portal as shown below.

![Starting the agent on the Foundry portal](./images/start-agent.png)

## Invoke hosted agents

You can view and test hosted agents in the agent playground UI. Hosted agents expose an OpenAI Responses-compatible API that you can invoke by using the Azure AI Projects SDK.

In [15]:
from azure.identity import AzureCliCredential # DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import AgentReference

# Initialize the client and retrieve the agent
client = AIProjectClient(endpoint=foundry_project_endpoint, credential=AzureCliCredential()) # DefaultAzureCredential()
agent = client.agents.get(agent_name=hosted_agent_name)
print(f"Retrieved agent: {agent.name} (version: {agent.versions.latest.version}")

# Get the OpenAI client and send a message
openai_client = client.get_openai_client()

response = openai_client.responses.create(
    input=[{"role": "user", "content": "Who is Houssem Dellai ?"}],
    # extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
    # extra_body={"agent": AgentReference(name=agent.name).as_dict()}
    extra_body={"agent": {"name": agent.name, "type": "agent_reference"}},
    # extra_body={"agent": AgentReference(name=agent.name, version=str(agent.version)).as_dict()}
)

print(f"Agent response: {response.output_text}")

Retrieved agent: foundry-hosted-agent (version: 2
Agent response: Houssem Dellai is a software developer, trainer, and content creator best known for his YouTube tutorials on **Microsoft technologies**, especially **Azure**, **.NET/ASP.NET Core**, **C#**, DevOps, and cloud architecture. He shares courses, demos, and best practices for building and deploying applications on Azure.


In [3]:
! curl -X POST http://localhost:8088/responses -H "Content-Type: application/json" -d '{"input": "Who is Houssem Dellai ?", "stream": false}'

{"error":"Invalid JSON payload: Expecting value: line 1 column 1 (char 0)"}


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed

  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100    83  100    75  100     8    351     37 --:--:-- --:--:-- --:--:--   387
curl: (3) URL rejected: Malformed input to a URL function
curl: (3) URL rejected: Port number was not a decimal number between 0 and 65535
curl: (3) unmatched close brace/bracket in URL position 6:
false}'
     ^


## Testing chat with Agent hosted in Container Apps

In [ ]:
aca_agent_endpoint = ! terraform output -raw aca_agent_endpoint
aca_agent_endpoint = aca_agent_endpoint.n
print("ACA Agent Endpoint: ", aca_agent_endpoint)

print(f'curl -X POST https://{aca_agent_endpoint}/responses -H "Content-Type: application/json" -d \'{{"input": "Who is Houssem Dellai ?", "stream": false}}\'')

In [ ]:
import requests

url = f"https://{aca_agent_endpoint}/responses"
headers = {"Content-Type": "application/json"}
payload = {"input": "Who is Houssem Dellai ?", "stream": False}

response = requests.post(url, headers=headers, json=payload)
print(response.status_code)
print(response.json())

## View container Log Stream

The container Logstream API for hosted agents gives you access to the system and console logs of the container deployed on your behalf in Microsoft's Azure environment to enable self-serve debugging for agent startup and runtime errors during deployment.

In [ ]:
# get console logs of the agent container
print(f'curl -N "{foundry_project_endpoint}/agents/{agent.name}/versions/{agent.version}/containers/default:logstream?kind=console&tail=500&api-version=2025-11-15-preview" \
             -H "Authorization: Bearer $(az account get-access-token --resource https://ai.azure.com --query accessToken -o tsv)"')

In [ ]:
# get system logs of the agent container
print(f'curl -N "{foundry_project_endpoint}/agents/{agent.name}/versions/{agent.version}/containers/default:logstream?kind=system&tail=500&api-version=2025-11-15-preview" \
             -H "Authorization: Bearer $(az account get-access-token --resource https://ai.azure.com --query accessToken -o tsv)"')

## More resources

- https://learn.microsoft.com/en-us/azure/ai-foundry/agents/concepts/hosted-agents?view=foundry